In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 55.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import pycountry

# Đường dẫn file trên Drive
file_path = "/content/drive/MyDrive/full_cities_130k.csv"

try:
    # --- TEST 1: Kiểm tra nạp file ---
    df = pd.read_csv(file_path, encoding='utf-8')
    print(f"✅ TEST 1: Đã nạp thành công {len(df)} địa điểm từ Google Drive!")

    # --- TEST 2: Kiểm tra sự tồn tại của cột country_code ---
    # (Nếu cột trong file của bạn tên khác, hãy sửa 'country_code' thành tên đó)
    column_name = 'country_code'
    if column_name not in df.columns:
        print(f"❌ TEST 2 Lỗi: Không tìm thấy cột '{column_name}'. Các cột hiện có: {list(df.columns)}")
    else:
        print(f"✅ TEST 2: Tìm thấy cột '{column_name}'.")

    # --- BƯỚC XỬ LÝ: CHUYỂN ĐỔI MÃ SANG TÊN QUỐC GIA ---
    def get_full_country_name(code):
        try:
            if pd.isna(code) or str(code).strip() == "":
                return "Unknown"
            country = pycountry.countries.get(alpha_2=str(code).upper())
            return country.name if country else code
        except:
            return code

    # Tạo cột mới 'country_full' từ cột 'country_code'
    df['country_full'] = df[column_name].apply(get_full_country_name)

    # --- TEST 3: Kiểm tra kết quả chuyển đổi ngẫu nhiên ---
    sample_size = 3
    print(f"✅ TEST 3: Đang kiểm tra {sample_size} mẫu chuyển đổi:")
    print(df[[column_name, 'country_full']].sample(sample_size).to_string(index=False))

    # --- TEST 4: Hiển thị 10 dòng đầu tiên của dữ liệu mới ---
    print("-" * 30)
    print("✅ TEST 4: 10 dòng dữ liệu đầu tiên sau khi xử lý:")
    # Hiển thị các cột quan trọng
    cols_to_show = ['name', column_name, 'country_full'] if 'name' in df.columns else df.columns[:4]
    print(df[cols_to_show].head(10).to_string(index=False))

except Exception as e:
    print(f"❌ Lỗi: {e}. Hãy kiểm tra lại đường dẫn hoặc cấu trúc file CSV.")

✅ TEST 1: Đã nạp thành công 166517 địa điểm từ Google Drive!
✅ TEST 2: Tìm thấy cột 'country_code'.
✅ TEST 3: Đang kiểm tra 3 mẫu chuyển đổi:
country_code  country_full
          CA        Canada
          US United States
          IT         Italy
------------------------------
✅ TEST 4: 10 dòng dữ liệu đầu tiên sau khi xử lý:
               name country_code country_full
               Vila           AD      Andorra
          El Tarter           AD      Andorra
Sant Julià de Lòria           AD      Andorra
       Santa Coloma           AD      Andorra
     Pas de la Casa           AD      Andorra
             Ordino           AD      Andorra
       les Escaldes           AD      Andorra
           Les Bons           AD      Andorra
         la Massana           AD      Andorra
             Encamp           AD      Andorra


In [4]:
import requests

def get_wikipedia_data(title):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"

    headers = {
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    r = requests.get(url, headers=headers)

    if r.status_code == 200:
        return r.json()
    else:
        return None

test = get_wikipedia_data("Manchester")
print(test["extract"])
print(test.get("thumbnail"))

Manchester is a city in Greater Manchester, England. It had a population of over 589,000 in 2024. It borders the Cheshire Plain to the south, the Pennines to the north and east, and the city of Salford to the west. The two cities and the surrounding towns form one of the United Kingdom's most populous conurbations, the Greater Manchester Built-up Area, which has a population of 2.87 million.
{'source': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3d/Deansgate_Square_-_geograph.org.uk_-_7803711.jpg/330px-Deansgate_Square_-_geograph.org.uk_-_7803711.jpg', 'width': 330, 'height': 181}


In [5]:
import json
import os
import pandas as pd
import requests
from tqdm import tqdm
import time
import pycountry

# --- BƯỚC 1: KIỂM TRA & CHUẨN BỊ CỘT QUỐC GIA (TEST 1) ---
def get_country_name(code):
    try:
        if pd.isna(code) or str(code).strip() == "": return "Unknown"
        # pycountry trả về đối tượng country, ta lấy thuộc tính .name
        country = pycountry.countries.get(alpha_2=str(code).upper())
        return country.name if country else code
    except:
        return code

# Đảm bảo df có cột country_full trước khi sample
if 'country_full' not in df.columns:
    # Lưu ý: Thay 'country_code' bằng tên cột mã quốc gia chính xác trong file của bạn
    col_code = 'country_code'
    df['country_full'] = df[col_code].apply(get_country_name)

print(f"✅ TEST 1: Đã chuẩn bị xong cột 'country_full'. (Ví dụ: VN -> {get_country_name('VN')})")

# --- BƯỚC 2: LẤY 10 MẪU VÀ KIỂM TRA (TEST 2) ---
df_test = df.sample(10).copy()
# SỬA LỖI .iloc: Thêm để lấy giá trị của hàng đầu tiên
sample_city = df_test['name'].iloc
print(f"✅ TEST 2: Đã chọn 10 thành phố ngẫu nhiên. Ví dụ: {sample_city}")

# --- BƯỚC 3: HÀM LẤY DỮ LIỆU WIKI ---
def get_wikipedia_data(city_name):
    title = str(city_name).replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    headers = {"User-Agent": "MyResearchBot/1.0 (duchb@example.com)"}
    try:
        r = requests.get(url, headers=headers, timeout=5)
        return r.json() if r.status_code == 200 else None
    except:
        return None

# --- BƯỚC 4: CHẠY TEST VÀ LƯU JSON ---
test_output_dir = "/content/drive/MyDrive/wiki_test_json/"
if not os.path.exists(test_output_dir):
    os.makedirs(test_output_dir)

results = []
print("\n🚀 Đang chạy TEST crawl 10 thành phố...")

# Duyệt theo hàng để lấy cả city và country cùng lúc
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    city_name = row["name"]
    country_name = row["country_full"]

    data = get_wikipedia_data(city_name)

    if data:
        # TEST 3: Kiểm tra khớp dữ liệu cho bản ghi đầu tiên thành công
        if len(results) == 0:
            print(f"✅ TEST 3: Khớp dữ liệu thành công: {city_name} -> {country_name}")

        results.append({
            "title": data.get("title"),
            "country": country_name, # Đưa quốc gia từ CSV vào đây
            "text": data.get("extract"),
            "image": data.get("thumbnail", {}).get("source") if data.get("thumbnail") else None,
            "url": data.get("content_urls", {}).get("desktop", {}).get("page")
        })
    time.sleep(0.1)

# Lưu file JSON
test_file = os.path.join(test_output_dir, "test_data_final.json")
with open(test_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

# TEST 4: XÁC NHẬN HOÀN TẤT
print(f"\n✅ TEST 4: HOÀN TẤT!")
print(f"- Lấy được: {len(results)}/10 thành phố.")
print(f"- File lưu tại: {test_file}")

# XEM TRƯỚC KẾT QUẢ (Đã sửa lỗi truy cập List)
if len(results) > 0:
    # results lấy phần tử đầu tiên (dictionary) trong danh sách
    print(f"\nXem thử bản ghi đầu tiên: {results['title']} | Quốc gia: {results['country']}")

✅ TEST 1: Đã chuẩn bị xong cột 'country_full'. (Ví dụ: VN -> Viet Nam)
✅ TEST 2: Đã chọn 10 thành phố ngẫu nhiên. Ví dụ: <pandas.core.indexing._iLocIndexer object at 0x7ad86728a080>

🚀 Đang chạy TEST crawl 10 thành phố...


  0%|          | 0/10 [00:00<?, ?it/s]

✅ TEST 3: Khớp dữ liệu thành công: Sabae -> Japan


100%|██████████| 10/10 [00:06<00:00,  1.53it/s]



✅ TEST 4: HOÀN TẤT!
- Lấy được: 10/10 thành phố.
- File lưu tại: /content/drive/MyDrive/wiki_test_json/test_data_final.json


TypeError: list indices must be integers or slices, not str

In [ ]:
import json
import os
import pandas as pd
import requests
from tqdm import tqdm
import time

# --- 1. CẤU HÌNH ---
output_dir = "/content/drive/MyDrive/wiki_data_json_final/"
os.makedirs(output_dir, exist_ok=True)
chunk_size = 1000  # Cứ duyệt đúng 1000 dòng trong CSV thì lưu 1 file

# --- 2. HÀM CRAWL ---
def get_wikipedia_data(city_name):
    title = str(city_name).replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    headers = {"User-Agent": "MyResearchBot/1.0 (duchb@example.com)"}
    try:
        r = requests.get(url, headers=headers, timeout=5)
        return r.json() if r.status_code == 200 else None
    except:
        return None

# --- 3. VÒNG LẶP THEO CHUNK (1000 DÒNG CSV) ---
total_rows = len(df)
print(f"🚀 Bắt đầu xử lý {total_rows} dòng từ CSV theo từng cụm 1000...")

for start_idx in range(0, total_rows, chunk_size):
    end_idx = min(start_idx + chunk_size, total_rows)
    file_num = (start_idx // chunk_size) + 1
    file_name = f"{output_dir}data_{file_num}.json"

    # --- KIỂM TRA ĐỂ CHẠY TIẾP ---
    if os.path.exists(file_name):
        # print(f"⏩ File {file_num} đã tồn tại (dòng {start_idx}-{end_idx}), bỏ qua.")
        continue

    print(f"\n📂 Đang xử lý file {file_num}: Dòng {start_idx} đến {end_idx}")
    results = []

    # Lấy đúng 1000 dòng từ CSV
    df_chunk = df.iloc[start_idx:end_idx]

    # Duyệt qua 1000 dòng này
    for _, row in tqdm(df_chunk.iterrows(), total=len(df_chunk), desc=f"File {file_num}"):
        city = row["name"]
        country = row.get("country_full", "Unknown")

        data = get_wikipedia_data(city)

        # Nếu lấy được dữ liệu từ Wiki thì mới thêm vào danh sách kết quả
        if data:
            results.append({
                "title": data.get("title"),
                "country": country,
                "text": data.get("extract"),
                "image": data.get("thumbnail", {}).get("source") if data.get("thumbnail") else None,
                "url": data.get("content_urls", {}).get("desktop", {}).get("page")
            })

        time.sleep(0.01) # Nghỉ ngắn

    # --- LƯU FILE SAU KHI DUYỆT HẾT 1000 DÒNG CSV ---
    # Ngay cả khi trong 1000 dòng chỉ lấy được 10 thành phố, vẫn lưu file để đánh dấu đã xong 1000 dòng này
    with open(file_name, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"✅ Đã lưu xong file {file_num}. Thành công {len(results)}/{chunk_size} thành phố.")

print("\n--- 🎉 TẤT CẢ ĐÃ HOÀN THÀNH ---")

🚀 Bắt đầu xử lý 166517 dòng từ CSV theo từng cụm 1000...

📂 Đang xử lý file 167: Dòng 166000 đến 166517


File 167:  24%|██▍       | 125/517 [00:24<01:12,  5.43it/s]

In [1]:
import json

# Đường dẫn đến file json của bạn
path = "/content/drive/MyDrive/wiki_data_json_final/data_2.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Kiểm tra thử dữ liệu
print(f"Đã tải xong! Số lượng bản ghi: {len(data)}")

Đã tải xong! Số lượng bản ghi: 863


In [2]:
import requests

def get_full_page_data(title):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|images",
        "explaintext": True,
        "titles": title,
        "imlimit": "max"
    }

    headers = {
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    r = requests.get(url, params=params, headers=headers)
    data = r.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))

    text = page.get("extract", "")

    image_titles = []
    if "images" in page:
        for img in page["images"]:
            if img["title"].lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                image_titles.append(img["title"])

    return text, image_titles

In [3]:
import requests
import time
def chunk_list(lst, size=50):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def get_image_urls(image_titles):
    if not image_titles:
        return []

    url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    image_urls = []

    # 🔥 chia thành batch 50
    for batch in chunk_list(image_titles, 50):

        params = {
            "action": "query",
            "format": "json",
            "prop": "imageinfo",
            "iiprop": "url",
            "titles": "|".join(batch)
        }

        r = requests.get(url, params=params, headers=headers)
        data = r.json()

        if "query" not in data:
            print("API error:", data)
            continue

        for page in data["query"]["pages"].values():
            if "imageinfo" in page:
                image_urls.append(page["imageinfo"][0]["url"])

        time.sleep(0.3)  # tránh rate limit

    return image_urls

In [4]:
# @title Test 1 cities
title = "New York City"

text, image_titles = get_full_page_data(title)
image_urls = get_image_urls(image_titles)



In [5]:
import os
import json
import time
from tqdm.notebook import tqdm

# 1. Cấu hình đường dẫn
input_folder = "/content/drive/MyDrive/wiki_data_json_final/"
output_folder = "/content/drive/MyDrive/wiki_results_final_json/"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Tự động đếm hoặc thiết lập số file tối đa bạn có
# Ví dụ bạn có 130 file data_1.json đến data_130.json
start_file = 1
end_file = 166

print(f"🚀 Bắt đầu xử lý FULL từ file {start_file} đến {end_file}")

# 2. Vòng lặp chạy chính xác theo số thứ tự
for i in range(start_file, end_file + 1):
    file_name = f"data_{i}.json"
    input_path = os.path.join(input_folder, file_name)
    output_path = os.path.join(output_folder, f"results_{i}.json")

    # Kiểm tra file đầu vào tồn tại
    if not os.path.exists(input_path):
        continue

    # Chế độ chạy tiếp (Resume)
    if os.path.exists(output_path):
        continue

    # --- Đọc file ---
    try:
        with open(input_path, "r", encoding="utf-8") as f:
            current_cities = json.load(f)
    except Exception as e:
        print(f"❌ Lỗi đọc file {file_name}: {e}")
        continue

    print(f"\n📂 File {i}/{end_file}: {file_name} ({len(current_cities)} bản ghi)")
    all_city_data = []

    # 3. Vòng lặp crawl cho từng thành phố
    for city_item in tqdm(current_cities, desc=f"Progress {file_name}", leave=False):
        title = city_item.get("title") or city_item.get("name")
        country = city_item.get("country", "Unknown")

        if not title:
            continue

        # Gọi hàm crawl (Đảm bảo đã chạy ô khai báo get_full_page_data và get_image_urls bản lấy FULL)
        result = get_full_page_data(title)
        if result is None:
            continue

        text, image_titles = result
        image_urls = get_image_urls(image_titles) # Lấy toàn bộ link ảnh

        all_city_data.append({
            "title": title,
            "country": country,
            "text": text,
            "images": image_urls,
            "wiki_url": f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
        })

        time.sleep(0.01)

    # 4. Lưu kết quả
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_city_data, f, ensure_ascii=False, indent=2)

    print(f"✅ Hoàn tất file {i}. Đã lưu {len(all_city_data)} bản ghi.")

print("\n🎉 TOÀN BỘ QUÁ TRÌNH HOÀN TẤT!")


🚀 Bắt đầu xử lý FULL từ file 1 đến 166

🎉 TOÀN BỘ QUÁ TRÌNH HOÀN TẤT!
